# Module 2: Scatter / Gather

This lab covers the one pattern in ParcelFlow beyond simple chaining: spreading a
list into per-item work and gathering the results back. It is the structure
underneath `map`/`reduce` and the array operations in scientific workflow systems.

**Learning objectives**
1. Spread a list into indexed parcels (`user[0]`, `user[1]`, ...).
2. See a per-item node run once per index *without writing a loop*.
3. Gather indexed results back into a single list.
4. Understand the strict **zip** semantics when a node requires two arrays.

> Note on parallelism: each per-item run is independent, which is the *structure*
> real systems parallelize. This engine runs them sequentially and deterministically.
> See the capstone exercise in Module 3 for adding actual concurrency.

In [ ]:
import sys, os
# Make the engine importable whether this notebook is launched from the
# repo root or from the education/ folder.
for candidate in ('.', '..'):
    if os.path.exists(os.path.join(candidate, 'workflow_engine.py')):
        sys.path.insert(0, os.path.abspath(candidate))
        break

from workflow_engine import WorkflowEngine
from base_node import BaseNode
from nodes import ArraySpreadNode, ProcessItemNode, CollectNode
print('ParcelFlow loaded.')

## 1. Scatter: a list becomes indexed parcels

`ArraySpreadNode` takes one parcel holding a list and emits one indexed parcel per
item, plus a `*_meta` parcel recording the length.

In [ ]:
engine = WorkflowEngine()
spread = ArraySpreadNode('spread', input_parcel='users', output_prefix='user')

final = engine.execute_workflow([spread], {'users': ['alice', 'bob', 'charlie']})

print('\nParcels after scatter:')
for name in sorted(final):
    print(' ', name, '=', final[name].value)

Notice the store now holds `user[0]`, `user[1]`, `user[2]` and a `user_meta`
parcel. Nothing has *processed* the items yet — we have only spread them.

## 2. Map: a per-item node runs once per index

`ProcessItemNode` declares `requires=['user']` — with **no index**. The engine sees
the indexed parcels and runs it once for each, passing the index in. Read its `run`
method: it handles a single item and never loops over the array.

In [ ]:
engine = WorkflowEngine()
nodes = [
    ArraySpreadNode('spread', input_parcel='users', output_prefix='user'),
    ProcessItemNode('process', input_prefix='user', output_prefix='processed'),
]
final = engine.execute_workflow(nodes, {'users': ['alice', 'bob', 'charlie']})

print('\nProcessed items:')
for name in sorted(final):
    if name.startswith('processed['):
        print(' ', name, '=', final[name].value)

## 3. Gather: collect the results back into a list

`CollectNode` uses the `*_meta` parcel to know how many items to expect, and only
runs once every `processed[i]` exists. This is the 'gather' (or 'reduce') step.

In [ ]:
engine = WorkflowEngine()
nodes = [
    ArraySpreadNode('spread', input_parcel='users', output_prefix='user'),
    ProcessItemNode('process', input_prefix='user', output_prefix='processed'),
    CollectNode('collect', input_prefix='processed', output_name='result', meta_parcel='user_meta'),
]
final = engine.execute_workflow(nodes, {'users': ['alice', 'bob', 'charlie']})

print('\nGathered result:', final['result'].value)

## 4. Zip semantics for two arrays

When a node requires *two* arrays, the engine runs only for indices present in
**both** — a strict zip, like Python's `zip()` stopping at the shorter list.

Below, `a` has three items and `b` has two. Predict how many times `ZipAddNode`
runs before you run the cell.

In [ ]:
class ZipAddNode(BaseNode):
    """Adds a[i] + b[i]; the engine supplies the index."""
    def __init__(self):
        super().__init__('zip_add', requires=['a', 'b'], outputs=['sum'])

    def run(self, parcels, index=None):
        a = parcels[f'a[{index}]'].value
        b = parcels[f'b[{index}]'].value
        return {f'sum[{index}]': a + b}

engine = WorkflowEngine()
initial = {'a[0]': 10, 'a[1]': 20, 'a[2]': 30, 'b[0]': 1, 'b[1]': 2}
final = engine.execute_workflow([ZipAddNode()], initial)

sums = {k: v.value for k, v in final.items() if k.startswith('sum[')}
print('\nSums produced:', sums)
print('Ran', len(sums), 'time(s) - index 2 of a has no matching b, so it is skipped.')

## Exercise: build your own scatter node

Implement `DoublerNode`: it requires the prefix `num` and, for each index, outputs
`doubled[i]` equal to twice the input number. Write it as if processing a single
item — let the engine handle the repetition.

Fill in the `run` method, then run the test cell below it.

In [ ]:
class DoublerNode(BaseNode):
    def __init__(self):
        super().__init__('doubler', requires=['num'], outputs=['doubled'])

    def run(self, parcels, index=None):
        # TODO:
        # 1. Read this index's value from parcels[f'num[{index}]'].value
        # 2. Return {f'doubled[{index}]': <twice that value>}
        pass

In [ ]:
# TEST CELL - do not edit. It should print 'PASS'.
engine = WorkflowEngine()
initial = {'num[0]': 5, 'num[1]': 11, 'num[2]': 0}
final = engine.execute_workflow([DoublerNode()], initial)
got = {k: v.value for k, v in final.items() if k.startswith('doubled[')}
expected = {'doubled[0]': 10, 'doubled[1]': 22, 'doubled[2]': 0}
assert got == expected, f'Not yet: got {got}, expected {expected}'
print('PASS')